# Traducción y Localización con Claude

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/casos-de-uso/05-traduccion-y-localizacion.ipynb)

Traducción con preservación de formato, localización cultural y pipeline multi-idioma.

Traducir texto plano es trivial. Traducir documentos Markdown, JSON de i18n o contenido técnico con términos propios, manteniendo el formato y adaptando las referencias culturales, es el verdadero reto. Este notebook cubre todas estas situaciones.

In [ ]:
%pip install anthropic langdetect --quiet

import os
import json
import asyncio
import time
import re
from typing import Optional
import anthropic

import nest_asyncio
nest_asyncio.apply()

MODELO = 'claude-haiku-4-5-20251001'
print(f'Usando modelo: {MODELO}')

## 1. Detección automática de idioma

Antes de traducir, detectamos el idioma fuente. `langdetect` funciona bien para textos de más de 20 palabras. Para textos cortos, usamos un prompt a Claude.

In [ ]:
from langdetect import detect, detect_langs

IDIOMAS = {
    'es': 'Español', 'en': 'English', 'fr': 'Français',
    'de': 'Deutsch', 'pt': 'Português', 'it': 'Italiano',
    'ja': '日本語', 'zh-cn': '中文', 'ar': 'العربية'
}

def detectar_idioma(texto: str) -> dict:
    """Detecta el idioma y devuelve código + nombre + confianza."""
    try:
        codigo = detect(texto)
        probabilidades = detect_langs(texto)
        confianza = next((p.prob for p in probabilidades if p.lang == codigo), 0.0)
        return {
            'codigo': codigo,
            'nombre': IDIOMAS.get(codigo, codigo),
            'confianza': round(confianza, 3)
        }
    except Exception:
        return {'codigo': 'unknown', 'nombre': 'Desconocido', 'confianza': 0.0}

textos_prueba = [
    'The quick brown fox jumps over the lazy dog.',
    'Le renard brun rapide saute par-dessus le chien paresseux.',
    'La inteligencia artificial está transformando el mundo.',
    'Die künstliche Intelligenz verändert die Welt.',
    'A inteligência artificial está mudando o mundo.'
]

print(f'{"Texto":<50} {"Idioma":<12} {"Confianza"}')
print('-' * 70)
for texto in textos_prueba:
    resultado = detectar_idioma(texto)
    print(f'{texto[:48]:<50} {resultado["nombre"]:<12} {resultado["confianza"]:.1%}')

## 2. TranslatorFormateado: preservación de Markdown

El mayor problema al traducir Markdown es que el modelo puede alterar la sintaxis (`**negrita**`, `[link](url)`, bloques de código). Usamos un prompt estructurado con instrucciones explícitas para preservar el formato.

In [ ]:
class TranslatorMarkdown:
    """Traduce texto Markdown preservando toda la sintaxis de formato."""

    SYSTEM_PROMPT = """Eres un traductor experto. Traduce el texto al idioma indicado.

REGLAS ESTRICTAS:
1. Preserva EXACTAMENTE toda la sintaxis Markdown: **negrita**, *cursiva*, `código`, [texto](url), # encabezados, tablas, listas
2. NO traduzcas: URLs, nombres de variables, bloques de código (```...```), nombres de funciones
3. Traduce solo el texto visible al lector
4. Devuelve SOLO el texto traducido, sin explicaciones ni prefijos
5. Mantén la estructura de párrafos y saltos de línea"""

    def __init__(self, api_key: Optional[str] = None):
        self.client = anthropic.Anthropic(api_key=api_key or os.environ.get('ANTHROPIC_API_KEY', 'demo'))

    def traducir(self, texto: str, idioma_destino: str,
                 glosario: Optional[dict] = None) -> str:
        """
        Traduce texto Markdown con glosario opcional de términos técnicos.

        Args:
            texto: texto Markdown a traducir
            idioma_destino: nombre del idioma ('English', 'Français', etc.)
            glosario: dict de términos que NO deben traducirse o tienen traducción fija
        """
        instrucciones_glosario = ''
        if glosario:
            terminos = '\n'.join(f'- "{k}" → "{v}"' for k, v in glosario.items())
            instrucciones_glosario = f'\n\nGLOSARIO (usa estas traducciones fijas):\n{terminos}'

        prompt = f"""Traduce al {idioma_destino}:{instrucciones_glosario}

---
{texto}
---"""

        respuesta = self.client.messages.create(
            model=MODELO,
            max_tokens=len(texto) * 2 + 500,
            system=self.SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return respuesta.content[0].text.strip()

# Ejemplo de texto Markdown con formato complejo
texto_markdown = """## Introducción a los **modelos de lenguaje**

Los LLMs (*Large Language Models*) como `claude-haiku-4-5-20251001` pueden:

- **Generar** texto coherente en múltiples idiomas
- *Resumir* documentos largos en segundos
- Responder preguntas sobre [documentación técnica](https://docs.anthropic.com)

| Modelo | Tokens/s | Coste |
|--------|----------|-------|
| Haiku  | ~100     | Bajo  |
| Sonnet | ~60      | Medio |

```python
# Este bloque NO debe traducirse
client.messages.create(model='claude-haiku-4-5-20251001')
```
"""

print('TranslatorMarkdown listo.')
print(f'Texto de ejemplo ({len(texto_markdown)} caracteres) preparado.')
# Para ejecutar (requiere API key):
# translator = TranslatorMarkdown()
# resultado = translator.traducir(texto_markdown, 'English', glosario={'modelos de lenguaje': 'language models'})
# print(resultado)

## 3. TranslatorFormateado: preservación de JSON (i18n)

Los archivos de internacionalización (i18n) en JSON tienen una estructura específica: solo deben traducirse los valores, nunca las claves.

In [ ]:
class TranslatorJSON:
    """Traduce archivos JSON de i18n preservando estructura y claves."""

    def __init__(self, api_key: Optional[str] = None):
        self.client = anthropic.Anthropic(api_key=api_key or os.environ.get('ANTHROPIC_API_KEY', 'demo'))

    def _extraer_valores(self, obj: dict, prefijo: str = '') -> dict:
        """Extrae todos los valores de texto de un JSON anidado."""
        resultado = {}
        for clave, valor in obj.items():
            clave_completa = f'{prefijo}.{clave}' if prefijo else clave
            if isinstance(valor, str):
                resultado[clave_completa] = valor
            elif isinstance(valor, dict):
                resultado.update(self._extraer_valores(valor, clave_completa))
        return resultado

    def _reconstruir(self, original: dict, traducciones: dict, prefijo: str = '') -> dict:
        """Reconstruye el JSON con las traducciones aplicadas."""
        resultado = {}
        for clave, valor in original.items():
            clave_completa = f'{prefijo}.{clave}' if prefijo else clave
            if isinstance(valor, str):
                resultado[clave] = traducciones.get(clave_completa, valor)
            elif isinstance(valor, dict):
                resultado[clave] = self._reconstruir(valor, traducciones, clave_completa)
            else:
                resultado[clave] = valor
        return resultado

    def traducir_json(self, datos: dict, idioma_destino: str) -> dict:
        """Traduce los valores de texto de un JSON de i18n."""
        valores = self._extraer_valores(datos)
        # Traduce todos los valores en una sola llamada (eficiente)
        valores_json = json.dumps(valores, ensure_ascii=False, indent=2)
        prompt = f"""Traduce al {idioma_destino} los VALORES del siguiente JSON.
NO cambies las claves. Preserva variables como {{{{name}}}}, {{{{count}}}}, etc.
Devuelve SOLO el JSON válido con los valores traducidos:

{valores_json}"""

        respuesta = self.client.messages.create(
            model=MODELO, max_tokens=len(valores_json) * 2 + 200,
            messages=[{'role': 'user', 'content': prompt}]
        )

        texto = respuesta.content[0].text.strip()
        # Extraer JSON de la respuesta
        match = re.search(r'\{.*\}', texto, re.DOTALL)
        if match:
            traducciones = json.loads(match.group())
        else:
            traducciones = json.loads(texto)

        return self._reconstruir(datos, traducciones)

# JSON de i18n de ejemplo
i18n_es = {
    "nav": {
        "inicio": "Inicio",
        "productos": "Productos",
        "contacto": "Contacto"
    },
    "mensajes": {
        "bienvenida": "Bienvenido, {{name}}",
        "carrito": "Tienes {{count}} artículos en el carrito",
        "error": "Ha ocurrido un error inesperado"
    }
}

print('JSON de i18n fuente:')
print(json.dumps(i18n_es, ensure_ascii=False, indent=2))
print('\nTranslatorJSON listo para traduccir a cualquier idioma.')
# Uso: translator = TranslatorJSON(); i18n_en = translator.traducir_json(i18n_es, 'English')

## 4. Localización cultural

La localización va más allá de la traducción: adapta fechas, formatos numéricos, unidades, referencias culturales y tono según el país destino.

In [ ]:
PERFILES_CULTURALES = {
    'es-ES': {
        'nombre': 'Español (España)',
        'formato_fecha': 'DD/MM/AAAA',
        'separador_decimal': ',',
        'moneda': 'EUR (€)',
        'tono': 'formal pero cercano',
        'notas': 'Usar vosotros, no ustedes. Precios en euros.'
    },
    'es-MX': {
        'nombre': 'Español (México)',
        'formato_fecha': 'DD/MM/AAAA',
        'separador_decimal': '.',
        'moneda': 'MXN (pesos)',
        'tono': 'amigable y directo',
        'notas': 'Usar ustedes. Adaptar expresiones locales mexicanas.'
    },
    'en-US': {
        'nombre': 'English (United States)',
        'formato_fecha': 'MM/DD/YYYY',
        'separador_decimal': '.',
        'moneda': 'USD ($)',
        'tono': 'casual and friendly',
        'notas': 'Use imperial units. Fahrenheit for temperature.'
    },
    'en-GB': {
        'nombre': 'English (United Kingdom)',
        'formato_fecha': 'DD/MM/YYYY',
        'separador_decimal': '.',
        'moneda': 'GBP (£)',
        'tono': 'polite and professional',
        'notas': 'Use British spellings (colour, behaviour). "Whilst" not "while".'
    }
}

def construir_prompt_localizacion(texto: str, locale: str) -> str:
    """Construye un prompt de localización con perfil cultural completo."""
    perfil = PERFILES_CULTURALES.get(locale, {})
    if not perfil:
        return f'Traduce al locale {locale}: {texto}'

    return f"""Localiza el siguiente texto para {perfil['nombre']}.

PERFIL CULTURAL:
- Formato de fecha: {perfil['formato_fecha']}
- Moneda: {perfil['moneda']}
- Tono: {perfil['tono']}
- Instrucciones específicas: {perfil['notas']}

Adapta (no solo traduzcas) el texto considerando el contexto cultural.
Devuelve SOLO el texto localizado:

{texto}"""

# Demostración del prompt generado
texto_origen = """¡Hola! Nuestros precios de verano están activos hasta el 4/7.
El termostato debe estar a unos 22 grados. ¡Aprovecha el descuento del 20%!"""

for locale in ['en-US', 'en-GB']:
    print(f'--- Prompt para {locale} ---')
    print(construir_prompt_localizacion(texto_origen, locale))
    print()

## 5. Pipeline async: 5 idiomas en paralelo

Traducimos un documento a 5 idiomas simultáneamente usando AsyncAnthropic.

In [ ]:
IDIOMAS_DESTINO = [
    {'codigo': 'en', 'nombre': 'English', 'locale': 'en-US'},
    {'codigo': 'fr', 'nombre': 'Français', 'locale': 'fr-FR'},
    {'codigo': 'de', 'nombre': 'Deutsch', 'locale': 'de-DE'},
    {'codigo': 'pt', 'nombre': 'Português', 'locale': 'pt-BR'},
    {'codigo': 'it', 'nombre': 'Italiano', 'locale': 'it-IT'}
]

async def traducir_idioma_async(cliente: anthropic.AsyncAnthropic,
                                 texto: str, idioma: dict) -> dict:
    """Traduce texto a un idioma usando AsyncAnthropic."""
    t0 = time.time()
    try:
        respuesta = await cliente.messages.create(
            model=MODELO,
            max_tokens=len(texto) * 2 + 200,
            system='Eres un traductor profesional. Devuelve SOLO el texto traducido.',
            messages=[{
                'role': 'user',
                'content': f'Traduce al {idioma["nombre"]}:\n\n{texto}'
            }]
        )
        return {
            'codigo': idioma['codigo'],
            'nombre': idioma['nombre'],
            'traduccion': respuesta.content[0].text.strip(),
            'latencia': round(time.time() - t0, 2),
            'tokens': respuesta.usage.input_tokens + respuesta.usage.output_tokens,
            'ok': True
        }
    except Exception as e:
        return {'codigo': idioma['codigo'], 'nombre': idioma['nombre'],
                'error': str(e), 'latencia': time.time() - t0, 'ok': False}

async def pipeline_multiidioma(texto: str) -> list[dict]:
    """Traduce un texto a 5 idiomas en paralelo."""
    async with anthropic.AsyncAnthropic() as cliente:
        tareas = [traducir_idioma_async(cliente, texto, idioma)
                  for idioma in IDIOMAS_DESTINO]
        resultados = await asyncio.gather(*tareas, return_exceptions=False)
    return resultados

# Texto de prueba
texto_origen = """La inteligencia artificial está transformando la forma en que trabajamos.
Los modelos de lenguaje permiten automatizar tareas complejas con resultados sorprendentes."""

print('Pipeline multi-idioma configurado para 5 idiomas en paralelo:')
for idioma in IDIOMAS_DESTINO:
    print(f'  → {idioma["nombre"]} ({idioma["codigo"]})')

print(f'\nTexto origen ({len(texto_origen)} chars):')
print(texto_origen)
print('\nEjecutar: resultados = asyncio.run(pipeline_multiidioma(texto_origen))')

## 6. Evaluación: BLEU-1 simplificado y LLM-judge

Evaluamos la calidad de las traducciones con dos métricas complementarias:
- **BLEU-1**: métrica léxica rápida (precision de unigramas)
- **LLM-judge**: evaluación semántica y cultural con Claude

In [ ]:
from collections import Counter

def bleu_1(hipotesis: str, referencia: str) -> float:
    """
    BLEU-1 simplificado: precisión de unigramas con brevity penalty.
    Para evaluación rápida de solapamiento léxico entre traducción y referencia.
    """
    hip_tokens = hipotesis.lower().split()
    ref_tokens = referencia.lower().split()

    if not hip_tokens:
        return 0.0

    ref_counter = Counter(ref_tokens)
    hits = sum(min(c, ref_counter[t]) for t, c in Counter(hip_tokens).items())
    precision = hits / len(hip_tokens)

    # Brevity penalty
    import math
    bp = 1.0 if len(hip_tokens) >= len(ref_tokens) else math.exp(1 - len(ref_tokens)/len(hip_tokens))
    return round(bp * precision, 4)


def llm_judge_prompt(original: str, traduccion: str, idioma_destino: str) -> str:
    """Prompt para evaluar calidad de traducción con Claude."""
    return f"""Evalúa esta traducción al {idioma_destino} en una escala del 1-10.

ORIGINAL:
{original}

TRADUCCIÓN:
{traduccion}

Evalúa: (1) fidelidad semántica, (2) naturalidad en el idioma destino,
(3) adecuación cultural, (4) preservación del tono.

Responde SOLO en formato JSON:
{{"puntuacion": 8.5, "fidelidad": 9, "naturalidad": 8, "cultura": 8, "tono": 9,
  "problemas": ["lista de problemas encontrados"], "sugerencia": "mejora principal"}}"""

# Demo de BLEU-1
pares_evaluacion = [
    ('The cat sat on the mat', 'The cat sat on the mat', 10.0),  # perfecta
    ('The cat sat on the mat', 'A cat is sitting on a rug', None),  # similar
    ('The cat sat on the mat', 'Dogs run in the park', None),  # diferente
]

print('Evaluación BLEU-1 (solapamiento léxico):')
print(f'{"Hipótesis":<35} {"Referencia":<35} BLEU-1')
print('-' * 80)
for hip, ref, _ in pares_evaluacion:
    score = bleu_1(hip, ref)
    print(f'{hip[:33]:<35} {ref[:33]:<35} {score:.4f}')

print('\nLLM-judge prompt generado para evaluación profunda.')
print('Ejemplo:')
print(llm_judge_prompt('Hola mundo', 'Hello world', 'English')[:200] + '...')

## Resumen y mejores prácticas

| Caso de uso | Clase/función | Consideración clave |
|---|---|---|
| Texto plano | `TranslatorMarkdown` con prompt simple | Especificar idioma destino claramente |
| Markdown complejo | `TranslatorMarkdown` | Prompt con reglas explícitas de formato |
| JSON de i18n | `TranslatorJSON` | Preservar variables `{{name}}` |
| Localización | `construir_prompt_localizacion` | Perfil cultural completo |
| Multi-idioma | `pipeline_multiidioma` | `AsyncAnthropic` + `gather` |
| Evaluación rápida | `bleu_1` | Para detección de regresiones |
| Evaluación profunda | `llm_judge_prompt` | Para validación de calidad |

**Coste estimado** (claude-haiku-4-5-20251001, texto de 200 palabras):
- 1 idioma: ~$0.0003
- 5 idiomas en paralelo: ~$0.0015  
- 100 documentos × 5 idiomas: ~$0.15

### Próximos pasos
- [Notebook Async Python para mayor rendimiento](../../python-para-ia/04-async-python-ia.ipynb)
- [Gestión de errores en APIs](../../apis/06-gestion-errores-resilience.ipynb)